# 21 Self-Labeled Data Check

In [1]:
import pandas as pd
from IPython.display import Audio, display

from woofdata.path import data_path, load_data_dir_from_env_file
from woofdata.tasks import task_rows
from woofdata.prepare.sources import self_labeled as sl

from woofdecoder.preprocess.media import TARGET_SAMPLE_RATE, cut_audio, load_audio_16k

In [2]:
load_data_dir_from_env_file()
raw_dir = data_path("raw/self_labeled")
COLLECTOR_DIR = data_path("raw/self_labeled/audio")
labels = pd.read_csv(COLLECTOR_DIR/"labels.csv")

In [3]:
for row in labels.itertuples():
    video_path = COLLECTOR_DIR/"videos"/f"{row.video_id}.mp4"
    
    audio = load_audio_16k(video_path, verbose=False)
    if audio is None:
        print(f"{row.video_id}: No audio track")
        continue
    
    span = cut_audio(audio, row.start_s, row.end_s)
    
    print(f"{row.video_id} {row.dog_id} {row.start_s}-{row.end_s}s ({len(span)/TARGET_SAMPLE_RATE:.1f}s)")
    display(Audio(span, rate=TARGET_SAMPLE_RATE))

A001 catersclips 48.0-78.0s (30.0s)


A002 whoknowsbruno 3.0-14.8s (11.8s)


A003 ericathescottishterrierexplorestheworld 0.0-6.0s (6.0s)


A004 rocketjax 6.0-18.0s (12.0s)


A005 lifeoflexithehusky 24.0-33.0s (9.0s)


A006 tikothetinychihuahua 6.0-12.0s (6.0s)


... (61 outputs left out)

A068 goldens 0.0-13.8s (13.8s)


A069 brunothebeagle27 0.0-6.8s (6.8s)


A070 themoriartyfamily 2.0-11.0s (9.0s)


A071 calypso'sinfo 12.0-17.0s (5.0s)


In [4]:
spans = []
for collection in sl.find_collections(raw_dir):
    spans.extend(sl.read_collection(raw_dir, collection))
sl.check_spans(raw_dir, spans)
split_by_dog = sl.assign_splits(spans)

print(len(spans), "spans |", sorted(split_by_dog.items()))
print("windows for a 25s span:", sl.window_bounds(25.0))
print("windows for a 10.4s span:", sl.window_bounds(10.4))

71 spans | [('adarshtiwarivlog', 'train'), ('animal&pets', 'train'), ('atishmaan', 'train'), ('bigshownggaybee', 'val'), ('bondivet', 'train'), ('bonnielonga', 'train'), ('briteeyedea', 'train'), ('brunothebeagle27', 'train'), ('bviral', 'train'), ("calypso'sinfo", 'train'), ('catersclips', 'train'), ('confidentpawsdogtraining', 'train'), ('cuteanimals', 'train'), ('cynthiamsoto', 'train'), ('debbiesteer', 'train'), ('doggyworld', 'val'), ('ericathescottishterrierexplorestheworld', 'train'), ('fluffydoesstuff', 'train'), ('funnycutepets', 'train'), ('gayavlogs', 'train'), ('gbccats', 'train'), ('goldens', 'train'), ('gunghodogtraining', 'train'), ('hollythebluestaffy', 'train'), ('ibisgar', 'val'), ('jawdroppingfacts11', 'train'), ('jawdroppingfacts12', 'train'), ('jeanieswan', 'train'), ('jeannehodson', 'train'), ('jrsodajerk', 'val'), ("julie'sk9academy", 'train'), ('kobethegoldenretriever', 'train'), ("leo'sadventure", 'train'), ('lifeoflexithehusky', 'train'), ('lifewithbibo', 'tra

In [5]:
for split in ("train", "val", "test"):
    rows = task_rows("audio_context", split)
    print(split, len(rows))
    print(rows.groupby(["source", "mapped_category"]).size().to_string(), "\n")

train 13065
source        mapped_category  
act_env       alert_aggressive     1824
              attention_seeking    2429
              other                3157
              playful_excited      5570
self_labeled  alert_aggressive        9
              attention_seeking      11
              distressed_lonely      46
              other                  10
              playful_excited         9 

val 442
source        mapped_category  
playpant      playful_excited      403
self_labeled  alert_aggressive       3
              attention_seeking      4
              distressed_lonely     29
              other                  2
              playful_excited        1 

Reminder: test is reported once, at the end. Tune on val.
test 357
source    mapped_category
playpant  playful_excited    357 



- Act_Env totals 12,980 (the 12,480 train + 500 validation clips), across 4 categories.
- distressed_lonely has 62 training windows and 13 val windows.
- Test is PlayPanT only.
- distressed_lonely is 62 of 13,042 rows (0.5%), from about 16 videos.

In [6]:
train = task_rows("audio_context", "train")
val = task_rows("audio_context", "val")
mine_train = train[train["source"] == "self_labeled"]
mine_val = val[val["source"] == "self_labeled"]

in_fusion = train["clip_id"].str.contains("/fusion/").sum() + val["clip_id"].str.contains("/fusion/").sum()

print("fusion rows in audio_context:", in_fusion)
print("dogs in both train and val:", set(mine_train["dog_id"]) & set(mine_val["dog_id"]))

fusion rows in audio_context: 0
dogs in both train and val: set()
